### 1. Preparação: Registrar Ferramentas no Unity Catalog
- O Agent Framework funciona melhor quando a LLM "chama" funções. Você deve transformar sua busca vetorial em uma Unity Catalog Function.

In [0]:
%sql
-- No SQL ou Python, registre a função de busca para o agente usar
CREATE OR REPLACE FUNCTION main.default.busca_normas_bacen(pergunta_usuario STRING)
RETURNS TABLE
RETURN SELECT texto_norma, link, tema 
       FROM VECTOR_SEARCH(
         index_name => "main.default.normativos_index", 
         query_text => pergunta_usuario, 
         num_results => 5
       );

### 2. Desenvolvimento: O Código do Agente (Python + LangChain)
- Você criará um arquivo Python (chain.py) que define o raciocínio do agente. O Databricks recomenda usar o MLflow para "empacotar" essa lógica.

In [0]:
import mlflow
from langchain_community.chat_models import ChatDatabricks
from langchain_genai import ChatGoogleGenerativeAI # Ou seu modelo corporativo

# Configura o modelo que você tem disponível (ex: databricks-dbrx-instruct)
llm = ChatDatabricks(endpoint="databricks-dbrx-instruct")

def agent_logic(input_text):
    # 1. Recupera normas relevantes usando a função do UC
    contexto = spark.sql(f"SELECT * FROM main.default.busca_normas_bacen('{input_text}')").collect()
    
    # 2. Constrói o Prompt (incluindo o contexto recuperado)
    prompt = f"""Você é um especialista em compliance do Banco Central.
    Analise o texto abaixo com base nestes normativos: {contexto}
    Texto para análise: {input_text}"""
    
    # 3. Chama a LLM
    return llm.predict(prompt)

# Registra o modelo no MLflow para o Databricks entender que isso é um AGENTE
mlflow.models.set_model(model=agent_logic)

### 3. Log e Registro no MLflow
- Para que o Databricks gerencie seu agente, você precisa "logar" o código no Unity Catalog. Isso transforma seu código em um modelo versionado.

In [0]:
with mlflow.start_run():
    logged_agent = mlflow.pyfunc.log_model(
        python_model="chain.py", # O arquivo que você criou acima
        artifact_path="agent",
        registered_model_name="main.default.agente_compliance_bacen"
    )

### 4. Deploy: Criar o "Serving Endpoint"
- Agora você transforma o modelo registrado em uma API ativa (Endpoint).
- No menu lateral do Databricks, vá em Machine Learning > Serving.
- Clique em Create Serving Endpoint.
- Selecione o modelo main.default.agente_compliance_bacen que você acabou de registrar.
- O Databricks criará uma interface de chat e uma API REST automaticamente.

### 5. Avaliação e Feedback (Agent Review App)
- Esta é a parte mais importante para o ambiente corporativo (Bancos).
- O Databricks gera um Review App: uma URL que você envia para os advogados ou analistas de compliance.
- Eles testam o agente e dão notas (positivo/negativo) e deixam comentários.
- Esses feedbacks são salvos em uma Delta Table de inferência, que você usa para ajustar o prompt ou o modelo de embedding (BERTimbau) depois.
- Por que seguir esse fluxo em vez de um script solto?
- Governança: Tudo (dados, vetores e o código do agente) está sob o Unity Catalog. Você sabe quem acessou o quê.
- Segurança: As credenciais e permissões de acesso às normas do BACEN são herdadas do usuário que está logado.
- Monitoramento: Você consegue ver exatamente quanto tempo cada busca vetorial demorou e qual foi o custo de tokens de cada análise de minuta.
- Dica para o seu projeto: Comece criando o endpoint de busca vetorial (Passo 1). Se ele retornar as normas certas, o resto do framework flui naturalmente. Você já tem permissão para criar "Serving Endpoints" no seu workspace do Databricks?